In [ ]:
import pandas as pd
import joblib

In [ ]:
import psycopg2 as psycopg
import os
from dotenv import load_dotenv

load_dotenv()

connection = {"sslmode": "require", "target_session_attrs": "read-write"}
postgres_credentials = {
    "host": "rc1b-uh7kdmcx67eomesf.mdb.yandexcloud.net",
    "port": "6432",
    "dbname": "playground_mle_20260614_fb07c65e05",
    "user": "mle_20260614_fb07c65e05",
    "password": os.environ.get("DB_DESTINATION_PASSWORD"),
}
assert all(
    [var_value != "" for var_value in list(postgres_credentials.values())]
)

connection.update(postgres_credentials)

# определяем название таблицы, в которой хранятся наши данные
TABLE_NAME = "users_churn"


# эта конструкция создаёт контекстное управление для соединения с базой данных
# оператор with гарантирует, что соединение будет корректно закрыто после выполнения всех операций с базой данных
# причём закрыто оно будет даже в случае ошибки при работе с базой данных
# это нужно, чтобы не допустить так называемую "утечку памяти"
with psycopg.connect(**connection) as conn:

    # создаём объект курсора для выполнения запросов к базе данных
    # с помощью метода execute() выполняется SQL-запрос для выборки данных из таблицы TABLE_NAME
    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {TABLE_NAME}")

        # извлекаем все строки, полученные в результате выполнения запроса
        data = cur.fetchall()

        # получаем список имён столбцов из объекта курсора
        columns = [col[0] for col in cur.description]

# создаём объект DataFrame из полученных данных и имён столбцов
# это позволяет удобно работать с данными в Python с использованием библиотеки Pandas
df = pd.DataFrame(data, columns=columns)

print(f"Размер нашей таблицы: {df.shape[0]} строк; {df.shape[1]} столбцов")

Размер нашей таблицы: 7043 строк; 22 столбцов


Инфиренс модели

In [13]:
# оценка качества модели
def evaluate_model():
    # загрузите результат прошлого шага: fitted_model.pkl
    with open("dvc/models/fitted_model.pkl", "rb") as fd:
        model = joblib.load(fd)

    X_test = pd.read_csv("dvc/split/X_val.csv")
    y_test = pd.read_csv("dvc/split/y_val.csv").squeeze()

    prediction = model.predict(X_test)
    probas = model.predict_proba(X_test)[:, 1]

    from sklearn.metrics import (
        roc_auc_score,
        precision_score,
        recall_score,
        f1_score,
        log_loss,
        confusion_matrix,
    )

    # импортируйте необходимые вам модули

    # заведите словарь со всеми метриками
    metrics = {}

    # посчитайте метрики из модуля sklearn.metrics
    # err_1 — ошибка первого рода
    # err_2 — ошибка второго рода
    _, err1, err2, _ = confusion_matrix(
        y_test, prediction, normalize="all"
    ).ravel()
    auc = roc_auc_score(y_test, probas)
    precision = precision_score(y_test, prediction)
    recall = recall_score(y_test, prediction)
    f1 = f1_score(y_test, prediction)
    logloss = log_loss(y_test, probas)

    # запишите значения метрик в словарь
    metrics["err1"] = err1
    metrics["err2"] = err2
    metrics["auc"] = auc
    metrics["precision"] = precision
    metrics["recall"] = recall
    metrics["f1"] = f1
    metrics["logloss"] = logloss

    print(f"X_test :\n {X_test.head(2).values.tolist()}")
    print(f"y_test: \n {y_test.head(2).values.tolist()}")

    print(f"Predicted values: {prediction[:20]}")
    print(f"Predicted probabilities: {probas[:20]}")
    print(f"True values: {y_test[:20].values}")


if __name__ == "__main__":
    evaluate_model()

X_test :
 [[438, '2014-02-01', 'Two year', 'Yes', 'Credit card (automatic)', 114.05, 8468.2, 'Fiber optic', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Male', 0, 'Yes', 'Yes', 'Yes'], [2280, '2014-02-01', 'Two year', 'Yes', 'Credit card (automatic)', 25.1, 1789.9, 'Fiber optic', 'No', 'No', 'No', 'No', 'No', 'No', 'Male', 1, 'No', 'No', 'Yes']]
y_test: 
 [0, 0]
Predicted values: [0 0 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 1 0 0]
Predicted probabilities: [0.01870995 0.01606406 0.04094973 0.42260813 0.05680502 0.76684427
 0.1984876  0.2300382  0.00930822 0.6600862  0.40638667 0.01333299
 0.27312199 0.85815072 0.01594397 0.18756193 0.1910203  0.73290991
 0.4276907  0.15088928]
True values: [0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0]


/Users/sergeyuser/Documents/Yandex Практикум/проекты/mlflow-3-logging/.venv/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.9.0 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/sergeyuser/Documents/Yandex Практикум/проекты/mlflow-3-logging/.venv/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.0 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/sergeyuser/Documents/Yandex Практикум/проекты/mlflow-3-logging/.venv/lib/pytho

In [ ]:
from sklearn.metrics import 
# импортируйте необходимые вам модули

# заведите словарь со всеми метриками
metrics = {}

# посчитайте метрики из модуля sklearn.metrics
# err_1 — ошибка первого рода
# err_2 — ошибка второго рода
_, err1, _, err2 = # ваш код здесь #
auc = # ваш код здесь #
precision = # ваш код здесь #
recall = # ваш код здесь #
f1 = # ваш код здесь #
logloss = # ваш код здесь #

# запишите значения метрик в словарь
metrics["err1"] = err1
metrics["err2"] = err2
metrics["auc"] = auc
metrics["precision"] = precision
metrics["recall"] = recall
metrics["f1"] = f1
metrics["logloss"] = logloss

In [ ]:
import os

import mlflow
import joblib
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

with open("dvc/models/fitted_model.pkl", "rb") as fd:
    model = joblib.load(fd)

X_test = pd.read_csv("dvc/split/X_val.csv")
y_test = pd.read_csv("dvc/split/y_val.csv").squeeze()

EXPERIMENT_NAME = "my_model_experiment"
RUN_NAME = "run1"
REGISTRY_MODEL_NAME = "churn_model_sergeyperminov"


os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
# os.environ["AWS_ACCESS_KEY_ID"] = "??"
# os.environ["AWS_SECRET_ACCESS_KEY"] = (
#    "??"
# )

mlflow.set_tracking_uri("http://127.0.0.1:5003")

pip_requirements = "requirements.txt"
prediction = model.predict(X_test)
signature = mlflow.models.infer_signature(X_test, prediction)
input_example = X_test[:10]
metadata = {"model_type": "monthly"}


experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="models",
        registered_model_name=REGISTRY_MODEL_NAME,
        pip_requirements=pip_requirements,
        signature=signature,
        input_example=input_example,
        metadata=metadata,
        await_registration_for=60,
        serialization_format="cloudpickle",
    )

/Users/sergeyuser/Documents/Yandex Практикум/проекты/mlflow-3-logging/.venv/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.9.0 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/sergeyuser/Documents/Yandex Практикум/проекты/mlflow-3-logging/.venv/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.0 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/sergeyuser/Documents/Yandex Практикум/проекты/mlflow-3-logging/.venv/lib/pytho

MlflowException: The sklearn model could not be serialized in the skops serialization format. skops does not support custom functions or classes that are not defined at the top level. To work around this limitation, you can set the serialization_format 'cloudpickle', while exercising caution due to the possible arbitrary code during model deserialization using CloudPickle.

In [ ]:
loaded_model = mlflow.sklearn.load_model(model_uri=model_info.model_uri)
model_predictions = loaded_model.predict(X_test)

assert model_predictions.dtype == int

print(model_predictions[:10])